In [7]:
import sympy as sp
from auxin_signalling_pathway.auxin_signalling_pathway import auxin_signalling_pathway as asp_class

asp_model = asp_class(nARFs=2, nIAAs=2)

print(asp_model)

{'arf__k': (1.0, 'production rate of ARFs'), 'arf__d': (1.0, 'decay rate of ARFs'), 'iaa__k': (1.0, 'production rate of Aux/IAAs'), 'iaa__d': (1.0, 'decay rate of Aux/IAAs'), 'k__D__ARF_1__ARF_1': (0.25, 'Association rate of dimer D__ARF_1__ARF_1'), 'd__D__ARF_1__ARF_1': (0.1, 'Dissociation rate of dimer D__ARF_1__ARF_1'), 'k__D__ARF_1__ARF_2': (0.25, 'Association rate of dimer D__ARF_1__ARF_2'), 'd__D__ARF_1__ARF_2': (0.1, 'Dissociation rate of dimer D__ARF_1__ARF_2'), 'k__D__ARF_1__iaa_1': (0.25, 'Association rate of dimer D__ARF_1__iaa_1'), 'd__D__ARF_1__iaa_1': (0.1, 'Dissociation rate of dimer D__ARF_1__iaa_1'), 'k__D__ARF_1__iaa_2': (0.25, 'Association rate of dimer D__ARF_1__iaa_2'), 'd__D__ARF_1__iaa_2': (0.1, 'Dissociation rate of dimer D__ARF_1__iaa_2'), 'k__D__ARF_2__ARF_2': (0.25, 'Association rate of dimer D__ARF_2__ARF_2'), 'd__D__ARF_2__ARF_2': (0.1, 'Dissociation rate of dimer D__ARF_2__ARF_2'), 'k__D__ARF_2__iaa_1': (0.25, 'Association rate of dimer D__ARF_2__iaa_1'), 

In [8]:
odes, free_variables, transition_rates, constraints = asp_model.generate_ode_system()


In [9]:
state_vars = list(asp_model.get_default_initial_conditions().keys())

promoter_vars = [s for s in state_vars if s[0]=='G']
promoter_odes = sp.Matrix([[ode for state_var, ode in zip(state_vars, odes) if str(state_var[0]) in promoter_vars]]).T
promoter_rhs_jac = sp.Matrix([promoter_odes]).T.jacobian(promoter_vars)
inhomog_part = sp.simplify(promoter_odes - promoter_rhs_jac @ sp.Matrix([promoter_vars]).T)

print(promoter_vars)
print(promoter_odes)
print(promoter_rhs_jac)
print(promoter_rhs_jac @ sp.Matrix([promoter_vars]).T)
print("inhomog", inhomog_part)

print(f"Checking conservation: {sp.simplify(sp.Matrix([[1 for _ in range(len(promoter_vars))]]) @ promoter_rhs_jac)}")



['G', 'G__ARF_1', 'G__ARF_1__ARF_1', 'G__ARF_1__ARF_2', 'G__ARF_1__iaa_1', 'G__ARF_1__iaa_2', 'G__ARF_2', 'G__ARF_2__ARF_1', 'G__ARF_2__ARF_2', 'G__ARF_2__iaa_1', 'G__ARF_2__iaa_2']
Matrix([[-ARF_1*G*k__G__ARF_1 - ARF_2*G*k__G__ARF_2 - D__ARF_1__ARF_1*G*k__G__ARF_1__ARF_1 - D__ARF_1__ARF_2*G*k__G__ARF_1__ARF_2 - D__ARF_1__ARF_2*G*k__G__ARF_2__ARF_1 - D__ARF_1__iaa_1*G*k__G__ARF_1__iaa_1 - D__ARF_1__iaa_2*G*k__G__ARF_1__iaa_2 - D__ARF_2__ARF_2*G*k__G__ARF_2__ARF_2 - D__ARF_2__iaa_1*G*k__G__ARF_2__iaa_1 - D__ARF_2__iaa_2*G*k__G__ARF_2__iaa_2 + G__ARF_1*d__G__ARF_1 + G__ARF_1__ARF_1*d__G__ARF_1__ARF_1 + G__ARF_1__ARF_2*d__G__ARF_1__ARF_2 + G__ARF_1__iaa_1*d__G__ARF_1__iaa_1 + G__ARF_1__iaa_2*d__G__ARF_1__iaa_2 + G__ARF_2*d__G__ARF_2 + G__ARF_2__ARF_1*d__G__ARF_2__ARF_1 + G__ARF_2__ARF_2*d__G__ARF_2__ARF_2 + G__ARF_2__iaa_1*d__G__ARF_2__iaa_1 + G__ARF_2__iaa_2*d__G__ARF_2__iaa_2], [ARF_1*G*k__G__ARF_1 - ARF_1*G__ARF_1*k__G__ARF_1__bind__ARF_1 - ARF_2*G__ARF_1*k__G__ARF_1__bind__ARF_2 - G__

In [ ]:
promoter_vars = [s for s in state_vars if s[0]=='G']
print(promoter_vars)
promoter_odes = sp.Matrix([[ode for state_var, ode in zip(state_vars, odes) if str(state_var[0]) in promoter_vars]]).T

inhomog_part_1 = promoter_odes.jacobian(promoter_vars)[1:, 0]

print(inhomog_part_1)

promoter_odes = promoter_odes[1:, :]
promoter_vars = promoter_vars[1:]


print(promoter_odes, promoter_vars)
promoter_odes = sp.Matrix([promoter_odes]).T.subs({'G': 1 - sum([sp.sympify(s) for s in promoter_vars])})
print(promoter_odes)
promoter_odes_jac = promoter_odes.jacobian(promoter_vars)
print(promoter_odes_jac)
# print(promoter_odes_jac.rank(), promoter_odes_jac.shape)

inhomog_part = sp.simplify(sp.Matrix(promoter_odes).T - promoter_odes_jac @ sp.Matrix(promoter_vars))

print('Inhomog', inhomog_part)
steady_state = promoter_odes_jac.solve(inhomog_part)
print("====")
print(f"steady state is {steady_state}")

['G', 'G__ARF_1', 'G__ARF_1__ARF_1', 'G__ARF_1__ARF_2', 'G__ARF_1__iaa_1', 'G__ARF_1__iaa_2', 'G__ARF_2', 'G__ARF_2__ARF_1', 'G__ARF_2__ARF_2', 'G__ARF_2__iaa_1', 'G__ARF_2__iaa_2']
Matrix([[ARF_1*k__G__ARF_1], [D__ARF_1__ARF_1*k__G__ARF_1__ARF_1], [D__ARF_1__ARF_2*k__G__ARF_1__ARF_2], [D__ARF_1__iaa_1*k__G__ARF_1__iaa_1], [D__ARF_1__iaa_2*k__G__ARF_1__iaa_2], [ARF_2*k__G__ARF_2], [D__ARF_1__ARF_2*k__G__ARF_2__ARF_1], [D__ARF_2__ARF_2*k__G__ARF_2__ARF_2], [D__ARF_2__iaa_1*k__G__ARF_2__iaa_1], [D__ARF_2__iaa_2*k__G__ARF_2__iaa_2]])
Matrix([[ARF_1*G*k__G__ARF_1 - ARF_1*G__ARF_1*k__G__ARF_1__bind__ARF_1 - ARF_2*G__ARF_1*k__G__ARF_1__bind__ARF_2 - G__ARF_1*d__G__ARF_1 - G__ARF_1*iaa_1*k__G__ARF_1__bind__iaa_1 - G__ARF_1*iaa_2*k__G__ARF_1__bind__iaa_2 + G__ARF_1__ARF_1*d__G__ARF_1__ARF_1__unbind__ARF_1 + G__ARF_1__ARF_2*d__G__ARF_1__ARF_2__unbind__ARF_2 + G__ARF_1__iaa_1*d__G__ARF_1__iaa_1__unbind__iaa_1 + G__ARF_1__iaa_2*d__G__ARF_1__iaa_2__unbind__iaa_2], [ARF_1*G__ARF_1*k__G__ARF_1__bind